In [1]:
import numpy as np
import torch

import os

from datasets.berlinurbangradient import BerlinUrbanGradient
from metrics import psnr, sa

from models import _jpeg2000

/data/mamba_envs/mfuchs/2025-compression/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/mamba_envs/mfuchs/2025-compression/lib/python3.13/site-packages/compressai/models/video/google.py:353: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @amp.autocast(enabled=False)
/data/mamba_envs/mfuchs/2025-compression/lib/python3.13/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
dataset_path = "./datasets/berlin-urban-gradient/"

jp2k_path = "/tmp/tmp.jp2"

In [3]:
test_dataset = BerlinUrbanGradient(dataset_path, split="test", transform=None)

psnr_metric = psnr.PeakSignalToNoiseRatio()
sa_metric = sa.SpectralAngle()

In [4]:
for cratio in [4, 8, 16, 27.75, 55.5, 111, 129.16, 253.71, 546.46, 1184.0]:
    cr_list = []
    psnr_list = []
    sa_list = []

    model = _jpeg2000.JPEG2000(target_compression_ratio=cratio, rescale=True)

    for data_org in test_dataset:
        data_org = data_org.unsqueeze(0)
        data_rec = model(data_org)

        size_raw = torch.Tensor.numpy(data_org).nbytes # / 1024 / 1024
        # print(f"raw size:\t{size_raw} MB")

        size_jp2k = os.path.getsize(jp2k_path) # / 1024 / 1024
        # print(f"jp2k size:\t{size_jp2k} MB")

        cr = size_raw / size_jp2k
        # print(f"CR:\t\t{cr}")

        psnr = psnr_metric(data_org, data_rec)
        # print(f"PSNR:\t\t{psnr} dB")

        sa = sa_metric(data_org, data_rec)
        # print(f"SA:\t\t{psnr} °")

        cr_list.append(cr)
        psnr_list.append(psnr)
        sa_list.append(sa)

    cr_avg = np.mean(cr_list)
    psnr_avg = np.mean(psnr_list)
    sa_avg = np.nanmean(sa_list)

    print(f"cr_avg:\t\t{cr_avg}")
    print(f"psnr_avg:\t{psnr_avg} dB")
    print(f"sa_avg: \t{sa_avg} °")
    print("===")

cr_avg:		4.000472419602268
psnr_avg:	73.64608764648438 dB
sa_avg: 	0.12102966755628586 °
===
cr_avg:		8.00066171781124
psnr_avg:	50.77434539794922 dB
sa_avg: 	1.3458786010742188 °
===
cr_avg:		16.00220810435303
psnr_avg:	38.79527282714844 dB
sa_avg: 	4.378114223480225 °
===
cr_avg:		25.999315432981028
psnr_avg:	33.954078674316406 dB
sa_avg: 	6.460806846618652 °
===
cr_avg:		54.004416147426994
psnr_avg:	29.397106170654297 dB
sa_avg: 	8.619270324707031 °
===
cr_avg:		109.95894226953017
psnr_avg:	26.532949447631836 dB
sa_avg: 	10.27907943725586 °
===
cr_avg:		127.96687219116342
psnr_avg:	26.046411514282227 dB
sa_avg: 	10.651535987854004 °
===
cr_avg:		251.6931691630532
psnr_avg:	24.166736602783203 dB
sa_avg: 	11.985115051269531 °
===
cr_avg:		544.681131336308
psnr_avg:	22.799592971801758 dB
sa_avg: 	12.664870262145996 °
===
cr_avg:		1179.38174064588
psnr_avg:	22.061893463134766 dB
sa_avg: 	12.257546424865723 °
===
